# Eksperimen Preprocessing BankChurners

Notebook ini digunakan untuk eksplorasi dataset BankChurners dan validasi alur preprocessing sebelum diotomatisasi pada `automate_amin.py`.

## 1. Import Library dan Load Dataset

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "preprocessing" else cwd
data_path = project_root / "BankChurners.csv"

df = pd.read_csv(data_path)
df.shape

## 2. Data Understanding

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 3. Data Quality Check

In [ ]:
quality_summary = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(),
    "dtype": df.dtypes.astype(str),
})

quality_summary

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_count

In [ ]:
categorical_columns = df.select_dtypes(include=["object", "category"]).columns
unknown_counts = {
    column: int((df[column] == "Unknown").sum())
    for column in categorical_columns
}

unknown_counts

## 4. Target Distribution

In [ ]:
target_counts = df["Attrition_Flag"].value_counts()
target_counts

In [ ]:
sns.countplot(data=df, x="Attrition_Flag")
plt.title("Distribusi Target Attrition_Flag")
plt.xlabel("Status Customer")
plt.ylabel("Jumlah Data")
plt.xticks(rotation=15)
plt.show()

## 5. Correlation Analysis

In [ ]:
leakage_columns = [
    "CLIENTNUM",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2",
]

corr_data = df.drop(columns=leakage_columns).copy()
corr_data["Attrition_Flag"] = corr_data["Attrition_Flag"].map({
    "Existing Customer": 0,
    "Attrited Customer": 1,
})

numeric_corr = corr_data.select_dtypes(include="number").corr()
sns.heatmap(numeric_corr, cmap="coolwarm", center=0)
plt.title("Korelasi Fitur Numerik")
plt.show()

## 6. Outlier Check

In [ ]:
numeric_columns = df.drop(columns=leakage_columns).select_dtypes(include="number").columns

outlier_summary = []
for column in numeric_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = ((df[column] < lower_bound) | (df[column] > upper_bound)).sum()
    outlier_summary.append({
        "column": column,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": int(outlier_count),
    })

pd.DataFrame(outlier_summary).sort_values("outlier_count", ascending=False)

## 7. Eksperimen Preprocessing

Alur berikut dibuat sama dengan `automate_amin.py`: drop kolom ID/leakage, encode target, pertahankan `Unknown` sebagai kategori valid, scaling fitur numerik, lalu one-hot encoding fitur kategorikal.

In [ ]:
target_column = "Attrition_Flag"
target_mapping = {
    "Existing Customer": 0,
    "Attrited Customer": 1,
}

processed = df.drop_duplicates().drop(columns=leakage_columns).copy()
processed[target_column] = processed[target_column].map(target_mapping).astype("int64")

features = processed.drop(columns=[target_column]).copy()
target = processed[target_column].copy()

categorical_features = list(features.select_dtypes(include=["object", "category"]).columns)
numeric_features = list(features.select_dtypes(include="number").columns)

features[numeric_features] = features[numeric_features].fillna(features[numeric_features].median())
features[numeric_features] = StandardScaler().fit_transform(features[numeric_features])
features[categorical_features] = features[categorical_features].fillna("Unknown")
features = pd.get_dummies(features, columns=categorical_features, dtype="int64")

processed_data = features.copy()
processed_data[target_column] = target

processed_data.head()

In [ ]:
processed_data.shape

In [ ]:
processed_data[target_column].value_counts()